In [ ]:
import numpy as np

In [ ]:
import pandas as pd
import csv

# Load data from CSV file
input_file = "/content/compiled_answers.csv"  # Update the path if needed
data = pd.read_csv(input_file)

algorithms_responses = {
    "Leiden": ["Leiden1", "Leiden2", "Leiden3"],
    "Louvain": ["Louvain1", "Louvain2", "Louvain3"],
    "Infomap": ["infomap1", "infomap2", "infomap3"],
    "Girvan-Newman": ["gn1", "gn2", "gn3"]
}
# Define questions in an array
questions = [
    "What are the overarching themes and ideas presented throughout the book?",
    "What are the relationships and interactions between major characters or entities across the book?",
    "What are the significant events or turning points in the book?"
]

metrics = [
    "comprehensiveness",
    "directness",
    "empowerment",
    "diversity"
]


# Extract algorithm response columns dynamically
algorithm_pairs = [
    ("Leiden", "Louvain"),
    ("Leiden", "Girvan-Newman"),
    ("Leiden", "Infomap"),
    ("Louvain", "Girvan-Newman"),
    ("Louvain", "Infomap"),
    ("Girvan-Newman", "Infomap")
]





In [ ]:
data.head()

,Leiden1,Leiden2,Leiden3,Louvain1,Louvain2,Louvain3,infomap1,infomap2,infomap3,gn1,gn2,gn3
0,The summary does not provide information on th...,The overarching themes and ideas presented thr...,The overarching themes and ideas presented thr...,The summary does not provide information on th...,The overarching themes and ideas presented thr...,The overarching themes and ideas presented thr...,The overarching themes and ideas presented thr...,"The book ""Metamorphosis"" presents overarching ...",The overarching themes and ideas presented thr...,The overarching themes and ideas presented thr...,The overarching themes and ideas presented thr...,The overarching themes and ideas presented thr...
1,"The major characters in the book ""Metamorphosi...","The book ""Metamorphosis"" by Franz Kafka featur...","The book ""Metamorphosis"" features a variety of...",The relationships and interactions between maj...,"The book ""Metamorphosis"" features a variety of...","The book ""Metamorphosis"" features a variety of...","In the book ""Metamorphosis"", the protagonist G...","The major characters in ""Metamorphosis"" are Gr...","In the book ""Metamorphosis"", the main characte...","The major characters in ""Metamorphosis"" have c...",The relationships and interactions between maj...,The relationships and interactions between maj...
2,The summary provided does not consistently giv...,"The summary of ""Metamorphosis"" does not provid...",The summary provided does not give specific in...,The summary provided does not offer specific d...,The summary does not provide specific informat...,The summary provided does not offer specific i...,The provided summary does not offer specific i...,The summary provided does not offer specific i...,The summary provided does not consistently giv...,"Based on the provided summaries, significant e...",The community summary does not provide specifi...,The summary does not provide detailed informat...


In [ ]:
responses_a = data.iloc[0][algorithms_responses["Leiden"]].tolist()
print(responses_a)

['The summary does not provide information on the overarching themes and ideas presented throughout the book "Metamorphosis". However, based on the community summary and other sources, the book explores themes such as work hierarchy, familial bonds, care and conflict within family relationships, the impact of external societal roles on an individual\'s life, and the emotional connection to personal spaces. It also delves into the hostility and aggression that can exist within family relationships, as seen in the interactions between Gregor and his father. Other themes include transformation, concern for loved ones, household responsibilities, and underlying issues within the family unit. The complex relationship between Gregor and his sister, as well as the family\'s struggle to deal with Gregor\'s transformation, highlight these themes. The book also explores themes of family responsibility, change and adaptation, and communication, as shown by Gregor\'s sister taking on the role of a

In [ ]:
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()


In [ ]:
# LLM evaluation function
def evaluate_metric(question, metric, answer_a, answer_b):

    prompt = (
        f"Question: {question}\n"
        f"Metric: {metric}\n"
        f"Answer A: {answer_a}\n"
        f"Answer B: {answer_b}\n\n"
        "Evaluate which answer is better according to the metric. "
        "Respond only with 'A', 'B', or 'TIE'. "
        "Ensure the response strictly starts with either 'A', 'B' or 'TIE'."
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    print(response)
    content = response.choices[0].message.content.strip()

    # Ensure response starts with 'A', 'B', or 'Tie'
    if not (content.startswith("A") or content.startswith("B") or content.startswith("TIE")):
        raise ValueError(
            f"Invalid response format: '{content}'. Ensure the response starts with 'A', 'B', or 'TIE'."
        )

    # Extract only 'A', 'B', or 'Tie'
    decision = content.split(":")[0].strip()
    if decision not in {"A", "B", "TIE"}:
        raise ValueError(f"Unexpected decision value: '{decision}'")

    return decision


In [ ]:
def calculate_win_rate(a_wins, b_wins, ties, total):
    if total > 0:
        return ((a_wins + 0.5 * ties) / total) * 100
    return 0.0

In [ ]:
# Run evaluations for each metric and repeat
results = {metric: [] for metric in metrics}
num_iterations = 5
# Loop through each metric

for metric in metrics:
    metric_results = []
    row = 0
    for question in questions:
        question_scores = {"Question": question, "Metric": metric}
        # Loop through defined algorithm pairs
        for algo_a, algo_b in algorithm_pairs:
            responses_a = data.iloc[row][algorithms_responses[algo_a]].values.tolist()  # Extract responses for algo_a
            responses_b = data.iloc[row][algorithms_responses[algo_b]].values.tolist()

            # Repeat the evaluation for stochasticity
            scores = []
            for i in range(num_iterations):
                try:
                    score = evaluate_metric(
                        question=question,
                        metric=metric,
                        answer_a=responses_a[i % len(responses_a)],
                        answer_b=responses_b[i % len(responses_b)]
                    )
                    scores.append(score)
                except ValueError as e:
                    print(f"Error evaluating {algo_a} vs {algo_b} for question '{question}': {e}")

            # Aggregate results (count A, B, Tie)
            print(f"{algo_a} vs {algo_b}")
            print(question)
            print(metric)
            A_wins = scores.count("A")
            B_wins = scores.count("B")
            ties = scores.count("TIE")
            total = A_wins + B_wins + ties
            print(f"A wins: {A_wins}, B wins: {B_wins}, Ties: {ties}, Total: {total}")

            # Calculate win rate with respect to algo_a
            win_rate_a = calculate_win_rate(A_wins, B_wins, ties, total)

            question_scores[f"{algo_a} vs {algo_b}"] = {
                "Win Rate A (%)": win_rate_a
            }
        row = row + 1
        metric_results.append(question_scores)
    results[metric] = metric_results

Leiden vs Louvain
What are the overarching themes and ideas presented throughout the book?
comprehensiveness
A wins: 4, B wins: 1, Ties: 0, Total: 5
Leiden vs Girvan-Newman
What are the overarching themes and ideas presented throughout the book?
comprehensiveness
A wins: 2, B wins: 3, Ties: 0, Total: 5
Leiden vs Infomap
What are the overarching themes and ideas presented throughout the book?
comprehensiveness
A wins: 0, B wins: 5, Ties: 0, Total: 5
Louvain vs Girvan-Newman
What are the overarching themes and ideas presented throughout the book?
comprehensiveness
A wins: 1, B wins: 4, Ties: 0, Total: 5
Louvain vs Infomap
What are the overarching themes and ideas presented throughout the book?
comprehensiveness
A wins: 1, B wins: 4, Ties: 0, Total: 5
Girvan-Newman vs Infomap
What are the overarching themes and ideas presented throughout the book?
comprehensiveness
A wins: 1, B wins: 4, Ties: 0, Total: 5
Leiden vs Louvain
What are the relationships and interactions between major character

In [ ]:
print(results)

{'comprehensiveness': [{'Question': 'What are the overarching themes and ideas presented throughout the book?', 'Metric': 'comprehensiveness', 'Leiden vs Louvain': {'Win Rate A (%)': 80.0}, 'Leiden vs Girvan-Newman': {'Win Rate A (%)': 40.0}, 'Leiden vs Infomap': {'Win Rate A (%)': 0.0}, 'Louvain vs Girvan-Newman': {'Win Rate A (%)': 0.0}, 'Louvain vs Infomap': {'Win Rate A (%)': 0.0}, 'Girvan-Newman vs Infomap': {'Win Rate A (%)': 20.0}}, {'Question': 'What are the relationships and interactions between major characters or entities across the book?', 'Metric': 'comprehensiveness', 'Leiden vs Louvain': {'Win Rate A (%)': 60.0}, 'Leiden vs Girvan-Newman': {'Win Rate A (%)': 100.0}, 'Leiden vs Infomap': {'Win Rate A (%)': 100.0}, 'Louvain vs Girvan-Newman': {'Win Rate A (%)': 60.0}, 'Louvain vs Infomap': {'Win Rate A (%)': 100.0}, 'Girvan-Newman vs Infomap': {'Win Rate A (%)': 100.0}}, {'Question': 'What are the significant events or turning points in the book?', 'Metric': 'comprehensive

In [ ]:
import csv

In [ ]:
output_file = "/content/metrics_comparison_scores.csv"
with open(output_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    # Write header row
    headers = ["Question", "Metric"] + [
        f"{algo_a} vs {algo_b} Win Rate A (%)" for algo_a, algo_b in algorithm_pairs
    ]
    writer.writerow(headers)

    # Write data rows
    for metric in metrics:
        for result in results[metric]:
            row = [result["Question"], result["Metric"]]
            for algo_a, algo_b in algorithm_pairs:
                comparison = result.get(f"{algo_a} vs {algo_b}", {"Win Rate A (%)": 0.0})
                row.append(f"{comparison['Win Rate A (%)']:.2f}")
            writer.writerow(row)

print(f"Results with unique rows saved to {output_file}")


Results with unique rows saved to /content/metrics_comparison_scores.csv
